<a href="https://colab.research.google.com/github/vblancoOR/econometria/blob/main/Autocorrelacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd #librería para manejo de datos

import numpy as np

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00381/PRSA_data_2010.1.1-2014.12.31.csv'

# Load the dataset into a pandas DataFrame
datos = pd.read_csv(url)
# Combine year, month, day, and hour into a single datetime column
datos['datetime'] = pd.to_datetime(data[['year', 'month', 'day', 'hour']])

# Set datetime as the index and drop original columns
datos.set_index('datetime', inplace=True)

# Drop unnecessary columns for simplicity
datos = datos[['pm2.5', 'TEMP', 'PRES', 'Iws']].dropna()
# Display the first few rows of the DataFrame
datos



In [ ]:

import statsmodels.api as sm

y=datos["pm2.5"]
X=datos[['TEMP', 'PRES', 'Iws']]

mco = sm.OLS(y, sm.add_constant(X)).fit()
print(mco.summary())


Gráficos

In [ ]:
import matplotlib.pylab as plt

plt.scatter(list(range(int(mco.nobs))), mco.resid)
plt.plot([-1,mco.nobs+1], [0,0], color='r')
plt.title("Observaciones vs Residuos")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Plot PM2.5 over time
data['pm2.5'].plot(figsize=(10, 6), title='PM2.5')
plt.xlabel('Tiempo')
plt.ylabel('PM2.5 (µg/m³)')
plt.show()

In [ ]:
retardo=1
residuos = mco.resid[retardo:]
residuos_retardados=mco.resid[:-retardo]

plt.scatter(residuos, residuos_retardados)
plt.show()

Durbin-Watson

In [ ]:
from statsmodels.stats.stattools import durbin_watson

dw = durbin_watson(mco.resid)
print("Durbin-Watson statistic:", dw)

H_Durbin

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

# Simulated example: Autoregressive data
np.random.seed(0)
n = 100
y_original = datos["pm2.5"]
y_retardada = np.roll(y_original, 1)  #
y_retardada[0] = 0  #


modelo = sm.OLS(y_original, sm.add_constant(y_retardada)).fit()

# Extract parameters
beta = modelo.params["x1"]  # Coeficiente
var_beta = modelo.bse["x1"] ** 2  # Variance del coeficiente

# Compute Durbin's h statistic
h = (1 - dw / 2) * np.sqrt(n / (1 - var_beta))


print("h-Durbin:", h)

# Check significance of h
if np.abs(h) > 1.96:  # Rough 95% confidence interval
    print("Autocorrelation detectada.")
else:
    print("Autocorrelation no significativa.")


Ljung-Box

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox


acorr_ljungbox(mco.resid, lags=5)


Corrección: Cochran-Orcutt

In [ ]:
rho= 1 - dw/2 # dw = 2(1-rho) => rho = 1 - DW/2
print(rho)
mco_autocorr=sm.GLSAR(y, sm.add_constant(X), rho=rho)
res=mco_autocorr.iterative_fit(maxiter=100,rtol=10**(-10))


print ('Iteraciones = %d -->  Converge: %s' % (res.iter, res.converged) )
print ('Rho =  ', mco_autocorr.rho)
print(res.summary())